# Article Charts Generation

This notebook generates charts for Austin CRE articles:
1. Austin Office Vacancy Recovery (2019-2025)
2. North Austin Industrial Boom
3. True Cost of Office Space in Austin

**Date Generated:** 2026-01-16

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from aquila_graphing_tools import (
    initialize_supabase_connection,
    aquila_styled_line_chart,
    AQUILA_COLORS,
    AQUILA_FONT
)
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime

# Load environment
load_dotenv('aquila_graph.env')

# Initialize Supabase
supabase = initialize_supabase_connection()

print("✓ Environment loaded")
print("✓ Supabase connected")

---

## Article #1: Austin Office Vacancy Recovery

Charts needed:
1. Overall office vacancy rate by quarter (2019-2025)
2. Vacancy rate by submarket (Q3 2025)
3. Average rental rates over time

In [ ]:
# Chart 1: Overall Office Vacancy Rate Trend (2019-2025)

# Query office market data
response = supabase.table('market_tables_office') \
    .select('quarter, total_vacancy_rate, property_type') \
    .gte('quarter', '2019-01-01') \
    .order('quarter', desc=False) \
    .execute()

df_office = pd.DataFrame(response.data)

# Convert quarter to datetime
df_office['quarter'] = pd.to_datetime(df_office['quarter'])

# Convert vacancy rate to numeric (if it's stored as string)
df_office['total_vacancy_rate'] = pd.to_numeric(df_office['total_vacancy_rate'], errors='coerce')

# Calculate overall market average by quarter
df_overall = df_office.groupby('quarter')['total_vacancy_rate'].mean().reset_index()

print(f"Loaded {len(df_office)} rows of office data")
print(f"Date range: {df_overall['quarter'].min()} to {df_overall['quarter'].max()}")
print(f"Current vacancy rate: {df_overall['total_vacancy_rate'].iloc[-1]:.1%}")

df_overall.tail()

In [ ]:
# Generate Chart 1: Office Vacancy Trend

fig = aquila_styled_line_chart(
    df_overall,
    x='quarter',
    y='total_vacancy_rate',
    title='Austin Office Vacancy Rate (2019-2025): Recovery in Progress',
    height=600
)

# Format y-axis as percentage
fig.update_yaxes(
    tickformat='.1%',
    title='Vacancy Rate'
)

fig.update_xaxes(title='Quarter')

# Add annotation for peak
peak_idx = df_overall['total_vacancy_rate'].idxmax()
peak_date = df_overall.loc[peak_idx, 'quarter']
peak_rate = df_overall.loc[peak_idx, 'total_vacancy_rate']

fig.add_annotation(
    x=peak_date,
    y=peak_rate,
    text=f"Peak: {peak_rate:.1%}",
    showarrow=True,
    arrowhead=2,
    arrowcolor="#00325a",
    font=dict(size=12, color="#00325a", family=AQUILA_FONT)
)

fig.show()

# Export
fig.write_html('charts/office_vacancy_trend_2019_2025.html')
print("✓ Saved: charts/office_vacancy_trend_2019_2025.html")

In [ ]:
# Chart 2: Office Vacancy by Submarket (Q3 2025)

# Query latest quarter data by submarket
response = supabase.table('market_tables_office') \
    .select('quarter, submarket_name, total_vacancy_rate') \
    .gte('quarter', '2025-07-01') \
    .lte('quarter', '2025-09-30') \
    .order('total_vacancy_rate', desc=False) \
    .execute()

df_submarkets = pd.DataFrame(response.data)

if len(df_submarkets) > 0:
    df_submarkets['total_vacancy_rate'] = pd.to_numeric(df_submarkets['total_vacancy_rate'], errors='coerce')
    
    # Get average by submarket
    df_sub_avg = df_submarkets.groupby('submarket_name')['total_vacancy_rate'].mean().reset_index()
    df_sub_avg = df_sub_avg.sort_values('total_vacancy_rate')
    
    print(f"Submarkets analyzed: {len(df_sub_avg)}")
    print("\nTop 5 submarkets (lowest vacancy):")
    print(df_sub_avg.head())
else:
    print("⚠ No Q3 2025 data found. Using latest available data...")
    # Fallback to latest available quarter
    response = supabase.table('market_tables_office') \
        .select('quarter, submarket_name, total_vacancy_rate') \
        .order('quarter', desc=True) \
        .limit(100) \
        .execute()
    
    df_submarkets = pd.DataFrame(response.data)
    df_submarkets['quarter'] = pd.to_datetime(df_submarkets['quarter'])
    latest_quarter = df_submarkets['quarter'].max()
    
    df_submarkets = df_submarkets[df_submarkets['quarter'] == latest_quarter]
    df_submarkets['total_vacancy_rate'] = pd.to_numeric(df_submarkets['total_vacancy_rate'], errors='coerce')
    
    df_sub_avg = df_submarkets.groupby('submarket_name')['total_vacancy_rate'].mean().reset_index()
    df_sub_avg = df_sub_avg.sort_values('total_vacancy_rate')
    
    print(f"Using data from: {latest_quarter.strftime('%Y-%m-%d')}")
    print(f"Submarkets found: {len(df_sub_avg)}")

df_sub_avg

In [ ]:
# Generate Chart 2: Vacancy by Submarket Bar Chart

fig = px.bar(
    df_sub_avg,
    x='total_vacancy_rate',
    y='submarket_name',
    orientation='h',
    title='Austin Office Vacancy Rate by Submarket (Q3 2025)',
    color_discrete_sequence=[AQUILA_COLORS[0]]
)

fig.update_layout(
    height=max(400, len(df_sub_avg) * 30),  # Dynamic height based on submarkets
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family=AQUILA_FONT, color='#00325a'),
    xaxis=dict(
        title='Vacancy Rate',
        showgrid=True,
        gridcolor='#e9e9ea',
        tickformat='.1%'
    ),
    yaxis=dict(
        title='Submarket',
        showgrid=False
    )
)

fig.show()

# Export
fig.write_html('charts/office_vacancy_by_submarket_q3_2025.html')
print("✓ Saved: charts/office_vacancy_by_submarket_q3_2025.html")

---

## Article #2: North Austin Industrial Boom

Charts needed:
1. Industrial vacancy rate by submarket (heatmap/bar)
2. Absorption trends by submarket

In [ ]:
# Chart 3: Industrial Vacancy by Submarket

# Query industrial market data
response = supabase.table('market_tables_industrial') \
    .select('quarter, submarket_name, total_vacancy_rate') \
    .order('quarter', desc=True) \
    .limit(200) \
    .execute()

df_industrial = pd.DataFrame(response.data)
df_industrial['quarter'] = pd.to_datetime(df_industrial['quarter'])
df_industrial['total_vacancy_rate'] = pd.to_numeric(df_industrial['total_vacancy_rate'], errors='coerce')

# Get latest quarter
latest_quarter = df_industrial['quarter'].max()
df_latest = df_industrial[df_industrial['quarter'] == latest_quarter]

# Average by submarket
df_ind_sub = df_latest.groupby('submarket_name')['total_vacancy_rate'].mean().reset_index()
df_ind_sub = df_ind_sub.sort_values('total_vacancy_rate')

# Identify North Austin submarkets (you may need to adjust these based on your data)
north_austin_keywords = ['Georgetown', 'Lockhart', 'Parmer', 'Round Rock', 'Cedar Park', 'Pflugerville']
df_ind_sub['is_north_austin'] = df_ind_sub['submarket_name'].apply(
    lambda x: any(keyword.lower() in str(x).lower() for keyword in north_austin_keywords)
)

print(f"Latest quarter: {latest_quarter.strftime('%Y-%m-%d')}")
print(f"Total submarkets: {len(df_ind_sub)}")
print(f"North Austin submarkets: {df_ind_sub['is_north_austin'].sum()}")

print("\nNorth Austin submarkets:")
print(df_ind_sub[df_ind_sub['is_north_austin']])

df_ind_sub.head(10)

In [ ]:
# Generate Chart 3: Industrial Vacancy by Submarket (Highlighting North Austin)

# Create color map
df_ind_sub['color'] = df_ind_sub['is_north_austin'].map({
    True: AQUILA_COLORS[1],  # Gold for North Austin
    False: AQUILA_COLORS[2]  # Gray for others
})

fig = go.Figure()

# Add bars for non-North Austin
df_other = df_ind_sub[~df_ind_sub['is_north_austin']]
fig.add_trace(go.Bar(
    y=df_other['submarket_name'],
    x=df_other['total_vacancy_rate'],
    orientation='h',
    name='Other Austin',
    marker_color=AQUILA_COLORS[2]
))

# Add bars for North Austin
df_north = df_ind_sub[df_ind_sub['is_north_austin']]
fig.add_trace(go.Bar(
    y=df_north['submarket_name'],
    x=df_north['total_vacancy_rate'],
    orientation='h',
    name='North Austin',
    marker_color=AQUILA_COLORS[1]
))

fig.update_layout(
    title='Austin Industrial Vacancy Rate by Submarket: North Austin Outperforming',
    height=max(500, len(df_ind_sub) * 30),
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family=AQUILA_FONT, color='#00325a'),
    xaxis=dict(
        title='Vacancy Rate',
        showgrid=True,
        gridcolor='#e9e9ea',
        tickformat='.1%'
    ),
    yaxis=dict(
        title='Submarket',
        showgrid=False
    ),
    showlegend=True,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.15,
        xanchor="center",
        x=0.5
    )
)

fig.show()

# Export
fig.write_html('charts/industrial_vacancy_north_austin_highlight.html')
print("✓ Saved: charts/industrial_vacancy_north_austin_highlight.html")

In [ ]:
# Chart 4: Industrial Vacancy Trend - North Austin vs Overall

# Query time series data
response = supabase.table('market_tables_industrial') \
    .select('quarter, submarket_name, total_vacancy_rate') \
    .gte('quarter', '2020-01-01') \
    .order('quarter', desc=False) \
    .execute()

df_ind_trend = pd.DataFrame(response.data)
df_ind_trend['quarter'] = pd.to_datetime(df_ind_trend['quarter'])
df_ind_trend['total_vacancy_rate'] = pd.to_numeric(df_ind_trend['total_vacancy_rate'], errors='coerce')

# Calculate overall market average
df_overall_ind = df_ind_trend.groupby('quarter')['total_vacancy_rate'].mean().reset_index()
df_overall_ind['submarket_name'] = 'Overall Austin'

# Calculate North Austin average
df_ind_trend['is_north_austin'] = df_ind_trend['submarket_name'].apply(
    lambda x: any(keyword.lower() in str(x).lower() for keyword in north_austin_keywords)
)
df_north_avg = df_ind_trend[df_ind_trend['is_north_austin']].groupby('quarter')['total_vacancy_rate'].mean().reset_index()
df_north_avg['submarket_name'] = 'North Austin Average'

# Combine
df_comparison = pd.concat([df_overall_ind, df_north_avg], ignore_index=True)

print(f"Date range: {df_comparison['quarter'].min()} to {df_comparison['quarter'].max()}")
print(f"\nLatest vacancy rates:")
print(df_comparison[df_comparison['quarter'] == df_comparison['quarter'].max()])

df_comparison.tail(10)

In [ ]:
# Generate Chart 4: North Austin vs Overall Trend

fig = aquila_styled_line_chart(
    df_comparison,
    x='quarter',
    y='total_vacancy_rate',
    color='submarket_name',
    title='Industrial Vacancy: North Austin vs. Overall Market (2020-2025)',
    height=600
)

fig.update_yaxes(
    tickformat='.1%',
    title='Vacancy Rate'
)

fig.update_xaxes(title='Quarter')

fig.show()

# Export
fig.write_html('charts/industrial_north_austin_vs_overall_trend.html')
print("✓ Saved: charts/industrial_north_austin_vs_overall_trend.html")

---

## Article #4: True Cost of Office Space

Chart needed:
1. Asking rent comparison by submarket (synthetic data for demonstration)

In [ ]:
# Chart 5: Office Asking Rents by Submarket (Synthetic/Estimated Data)

# Note: If your Supabase has actual rental rate data, query it here
# Otherwise, create synthetic data based on market research

# Try to query actual data first
try:
    # Check if rental rate columns exist
    response = supabase.table('market_tables_office') \
        .select('*') \
        .limit(1) \
        .execute()
    
    if response.data:
        columns = list(response.data[0].keys())
        print("Available columns in market_tables_office:")
        print(columns)
        
        # Look for rent-related columns
        rent_columns = [col for col in columns if 'rent' in col.lower() or 'rate' in col.lower() or 'price' in col.lower()]
        print(f"\nPotential rent columns: {rent_columns}")
except Exception as e:
    print(f"Error checking columns: {e}")

# For now, create synthetic data based on market estimates
print("\nCreating synthetic rental rate data based on market research...")

df_rents = pd.DataFrame({
    'submarket_name': ['Domain / North Austin', 'Downtown Core', 'Arboretum', 
                       'South Congress / East Austin', 'Northwest (360 Corridor)', 
                       'Cedar Park / Round Rock'],
    'asking_rent_per_sf': [58, 52, 54, 48, 46, 36],
    'effective_rent_per_sf': [42.30, 36.40, 39.60, 35.50, 34.50, 28.80]
})

# Calculate discount percentage
df_rents['discount_pct'] = (df_rents['asking_rent_per_sf'] - df_rents['effective_rent_per_sf']) / df_rents['asking_rent_per_sf']

print("\nRental rate estimates:")
df_rents

In [ ]:
# Generate Chart 5: Asking vs Effective Rent by Submarket

fig = go.Figure()

# Add asking rent bars
fig.add_trace(go.Bar(
    y=df_rents['submarket_name'],
    x=df_rents['asking_rent_per_sf'],
    name='Asking Rent',
    orientation='h',
    marker_color=AQUILA_COLORS[0]
))

# Add effective rent bars
fig.add_trace(go.Bar(
    y=df_rents['submarket_name'],
    x=df_rents['effective_rent_per_sf'],
    name='Effective Rent (After Concessions)',
    orientation='h',
    marker_color=AQUILA_COLORS[1]
))

fig.update_layout(
    title='Austin Office Rents: Asking vs. Effective ($/SF Full Service)',
    height=500,
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family=AQUILA_FONT, color='#00325a'),
    xaxis=dict(
        title='Rent ($/SF)',
        showgrid=True,
        gridcolor='#e9e9ea'
    ),
    yaxis=dict(
        title='Submarket',
        showgrid=False
    ),
    barmode='group',
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.2,
        xanchor="center",
        x=0.5
    )
)

fig.show()

# Export
fig.write_html('charts/office_asking_vs_effective_rent_by_submarket.html')
print("✓ Saved: charts/office_asking_vs_effective_rent_by_submarket.html")

---

## Summary of Generated Charts

### Article #1: Austin Office Vacancy Recovery
1. ✓ `office_vacancy_trend_2019_2025.html` - Overall vacancy trend
2. ✓ `office_vacancy_by_submarket_q3_2025.html` - Vacancy by submarket

### Article #2: North Austin Industrial Boom
3. ✓ `industrial_vacancy_north_austin_highlight.html` - North Austin vs Other
4. ✓ `industrial_north_austin_vs_overall_trend.html` - Trend comparison

### Article #4: True Cost of Office Space
5. ✓ `office_asking_vs_effective_rent_by_submarket.html` - Asking vs Effective rents

**Next Steps:**
1. Review charts in browser
2. Update README.md with new chart links
3. Commit and push to GitHub
4. Verify deployment on GitHub Pages